# 3. Closure tests

In a closure test, the "experimental data" are model output at known parameters, the
**pseudo-data**. The Bayesian analysis with the emulator should then recover these parameters.
This tests the whole chain of design, emulator and sampler before it is applied to real data,
and it shows how the emulators affect the posterior.

The pseudo-data are the model values at validation points of notebook 2, which were not used in
the training. They get the uncertainty of 6.59% of the paper, without random fluctuations.
The posterior should then contain the true parameters, and its maximum should be close to them;
deviations come from the emulator.

The posterior is sampled with pocoMC (`BayesianAnalysis.run_pocomc`) for each emulator trained on
the log of the observables in notebook 2 (`output/emulators/*_log.dill`).

In [ ]:
import logging
import pickle
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter

from gpbayestools import load_emulator, parse_model_parameter_file
from gpbayestools.bayesian_analysis import BayesianAnalysis

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings("ignore")

parameter_file = "data/parameters_DIS.txt"
emulator_dir = Path("output/emulators")
closure_dir = Path("output/closure")
figure_dir = Path("figures")

emulator_names = ["Scikit-learn", "PCGP", "PCSK", "HetGP", "SparseGP"]
emulator_files = {name: emulator_dir / f"{name}_log.dill" for name in emulator_names}

pardict = parse_model_parameter_file(parameter_file)
labels = [val[0] for val in pardict.values()]
n_train = 500

## Pseudo-data

The closure points are five validation points: the one closest to the maximum a posteriori
(MAP) parameters of the HERA fit in the paper, $Q_{s,0} = 0.393$ GeV and $\lambda = 0.219$, and
four random ones. The emulators return predictions in log space, so the pseudo-data are
log-transformed as well: the values are $\log y$ and the uncertainties are the relative
uncertainties $\sigma/y = 0.0659$ (see the `BayesianAnalysis` docstring).

The experimental data file contains a dictionary with one entry, whose `"obs"` array holds the
values and the uncertainties, like one point of the training data.

In [ ]:
# the model values of all design points, in log space
emu = load_emulator(emulator_files["PCGP"])
design, log_values, rel_errors = emu.design_points, emu.model_data, emu.model_data_err

validation = np.arange(n_train, len(design))
paper_map = np.array([0.393, 0.219])
scale = np.array([val[2] - val[1] for val in pardict.values()])
closest = validation[np.argmin(np.sum(((design[validation] - paper_map) / scale) ** 2, axis=1))]
rng = np.random.default_rng(1)
others = rng.choice(validation[validation != closest], size=4, replace=False)
closure_points = np.concatenate([[closest], others])

exp_files = {}
for k, i in enumerate(closure_points):
    path = closure_dir / f"point_{k}" / "exp_data.pkl"
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump({"pseudo_data": {"obs": np.vstack([log_values[i], rel_errors[i]])}}, f)
    exp_files[k] = path
    print(f"point {k}: design point {i}, true parameters {design[i]}")
truth = design[closure_points]

## Sampling the posteriors

For each closure point and emulator, a `BayesianAnalysis` is created with the pseudo-data, the
emulator is loaded and pocoMC samples the posterior with a uniform prior in the parameter ranges.
The likelihood is Gaussian with the sum of the emulator covariance and the covariance of the
pseudo-data. Each chain is written to `output/closure/point_<k>/<emulator>/chain_pocomc.pkl`.
The pocoMC settings are smaller than the defaults, which is sufficient for two parameters.
This takes about 15 minutes on a laptop.

In [ ]:
pocomc_settings = {
    "n_effective": 512,
    "n_active": 256,
    "n_prior": 1024,
    "n_total": 2000,
    "n_evidence": 0,
    "seed": 1,
}

posteriors = {}
run_times = {}
for k in exp_files:
    for name in emulator_names:
        analysis = BayesianAnalysis(
            exp_files[k],
            parameter_file,
            mcmc_path=closure_dir / f"point_{k}" / name / "chain.pkl",
        )
        analysis.load_emulators([emulator_files[name]])
        start = time.perf_counter()
        analysis.run_pocomc(**pocomc_settings)
        run_times[k, name] = time.perf_counter() - start
        posteriors[k, name] = analysis.chain
    print(f"point {k} done")

## Posteriors at the point closest to the MAP of the paper

The contours enclose 68% and 95% of the posterior samples (from a smoothed histogram). The black cross marks the true
parameters of the pseudo-data.

In [ ]:
colors = dict(zip(emulator_names, plt.rcParams["axes.prop_cycle"].by_key()["color"]))


def contour_levels(hist, fractions=(0.95, 0.68)):
    # density levels that enclose the given fractions of the samples
    h = np.sort(hist.ravel())[::-1]
    cumulative = np.cumsum(h) / h.sum()
    return [h[np.searchsorted(cumulative, f)] for f in fractions]


def plot_posteriors(k, filename=None):
    fig, axes = plt.subplots(2, 2, figsize=(8, 7))
    ranges = [(val[1], val[2]) for val in pardict.values()]
    samples_all = np.concatenate([posteriors[k, name] for name in emulator_names])
    # zoom to the region of the posteriors
    lo, hi = np.percentile(samples_all, [0.1, 99.9], axis=0)
    pad = 0.3 * (hi - lo)
    lims = [(max(r[0], l - p), min(r[1], h + p)) for r, l, h, p in zip(ranges, lo, hi, pad)]
    for name in emulator_names:
        s = posteriors[k, name]
        for d in range(2):
            axes[d, d].hist(
                s[:, d], bins=40, range=lims[d], density=True, histtype="step",
                color=colors[name], label=name,
            )
        hist, xe, ye = np.histogram2d(s[:, 0], s[:, 1], bins=40, range=lims)
        # smooth the histogram of the finite number of samples
        hist = gaussian_filter(hist, 1.5)
        xc, yc = 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])
        axes[1, 0].contour(
            xc, yc, hist.T, levels=sorted(contour_levels(hist)), colors=colors[name],
            linewidths=1.2,
        )
    for d in range(2):
        axes[d, d].axvline(truth[k, d], color="k", ls="--")
        axes[d, d].set_xlim(lims[d])
        axes[d, d].set_yticks([])
    axes[1, 0].plot(*truth[k], "kx", ms=10, mew=2)
    axes[1, 0].set_xlim(lims[0])
    axes[1, 0].set_ylim(lims[1])
    axes[1, 0].set_xlabel(labels[0])
    axes[1, 0].set_ylabel(labels[1])
    axes[1, 1].set_xlabel(labels[1])
    axes[0, 1].axis("off")
    axes[0, 1].legend(*axes[0, 0].get_legend_handles_labels(), loc="center")
    fig.tight_layout()
    if filename is not None:
        fig.savefig(figure_dir / filename, dpi=150)


plot_posteriors(0, "03_posterior_closure_point.png")

## Recovery of the true parameters at all closure points

For each parameter, the deviation of the posterior mean from the truth is shown in units of the
posterior standard deviation, $(\bar\theta - \theta_\mathrm{true}) / \sigma_\theta$. Values within
about $\pm 2$ mean that the truth is recovered. Since the pseudo-data have no random fluctuations,
the deviation is mostly due to the emulator; the posterior width is set by the 6.59% uncertainty.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
width = 0.15
for j, name in enumerate(emulator_names):
    pulls = np.array([
        (posteriors[k, name].mean(axis=0) - truth[k]) / posteriors[k, name].std(axis=0)
        for k in exp_files
    ])
    for d in range(2):
        axes[d].bar(
            np.arange(len(exp_files)) + (j - 2) * width, pulls[:, d], width,
            color=colors[name], label=name,
        )
for d in range(2):
    axes[d].axhline(0, color="k", lw=0.8)
    for y in (-2, 2):
        axes[d].axhline(y, color="k", ls=":", lw=0.8)
    axes[d].set_xticks(np.arange(len(exp_files)))
    axes[d].set_xlabel("closure point")
    axes[d].set_title(labels[d])
axes[0].set_ylabel("(posterior mean - truth) / posterior std")
axes[1].legend(fontsize=9, ncols=2)
fig.tight_layout()
fig.savefig(figure_dir / "03_closure_pulls.png", dpi=150)

In [ ]:
print(f"{'point':>5s} {'emulator':12s} " + " ".join(f"{p:>22s}" for p in pardict) + f" {'time [s]':>9s}")
for k in exp_files:
    for name in emulator_names:
        s = posteriors[k, name]
        cols = " ".join(
            f"{m:9.4f} +- {sd:6.4f} ({t:.3f})"
            for m, sd, t in zip(s.mean(axis=0), s.std(axis=0), truth[k])
        )
        print(f"{k:5d} {name:12s} {cols} {run_times[k, name]:9.1f}")

The table lists the posterior mean ± standard deviation and, in parentheses, the true value.

## Posterior predictive distribution

The posterior predictive distribution shows how well the calibrated model describes the
(pseudo-)data. At 200 posterior samples of the first closure point, the emulators predict the
observables, which are shown relative to the pseudo-data. The grey band is the 6.59%
uncertainty of the pseudo-data.

In [ ]:
k = 0
rng = np.random.default_rng(2)
fig, axes = plt.subplots(len(emulator_names), 1, figsize=(9, 2.0 * len(emulator_names)), sharex=True, sharey=True)
data = np.exp(log_values[closure_points[k]])
obs_index = np.arange(len(data))
for ax, name in zip(axes, emulator_names):
    emu = load_emulator(emulator_files[name])
    samples = posteriors[k, name]
    samples = samples[rng.choice(len(samples), size=200, replace=False)]
    ratio = np.exp(emu.predict(samples, return_cov=False)) / data
    lo, med, hi = np.percentile(ratio, [5, 50, 95], axis=0)
    ax.fill_between(obs_index, 1 - rel_errors[0], 1 + rel_errors[0], color="0.85", label="pseudo-data uncertainty")
    ax.fill_between(obs_index, lo, hi, color=colors[name], alpha=0.5, label="90% posterior predictive")
    ax.plot(obs_index, med, color=colors[name], lw=1)
    ax.axhline(1, color="k", lw=0.8)
    ax.text(0.01, 0.85, name, transform=ax.transAxes)
axes[-1].set_xlabel("observable index (HERA data point)")
axes[len(axes) // 2].set_ylabel("prediction / pseudo-data")
axes[0].legend(loc="upper right", ncols=2, fontsize=8)
fig.tight_layout()
fig.savefig(figure_dir / "03_posterior_predictive.png", dpi=150)

## Discussion

- **Recovery of the truth.** All emulators recover the true parameters at all closure points,
  the posterior means deviate from the truth by less than about 0.2 standard deviations. Since
  the pseudo-data are noise-free model values, this deviation is caused by the emulator, and it
  is small because the emulator errors (about 0.5%, notebook 2) are much smaller than the 6.59%
  uncertainty. With fluctuated pseudo-data, like a real measurement, the posterior means would
  scatter by about one standard deviation around the truth.
- **Posterior widths.** Although the data uncertainty is 6.59%, the combination of the 100
  data points constrains $Q_{s,0}$ and $\lambda$ to about 2–3%. The posteriors of Scikit-learn,
  PCGP, PCSK and HetGP are practically identical. The posteriors of SparseGP are 10–40% wider,
  since its overestimated uncertainty enters the covariance of the likelihood. A conservative
  emulator widens the posterior, an overconfident one would make it too narrow, which matters
  more when the emulator uncertainty is comparable to the data uncertainty.
- **Correlation.** $Q_{s,0}$ and $\lambda$ are strongly anticorrelated: in the $x$ range of the
  data, a larger $Q_{s,0}$ can be compensated by a smaller $\lambda$ in the saturation scale
  $Q_s^2(x) = Q_{s,0}^2 x^{-\lambda} (1-x)$.
- **Posterior predictive.** The emulator predictions at the posterior samples describe the
  pseudo-data within about 2%, well inside the 6.59% uncertainty; for SparseGP, the wider
  posterior gives a wider band.

For the analysis of real data, the pseudo-data are replaced by the measured values (for
`log_trafo=True` emulators log-transformed as above), and the posterior can be sampled with
`run_pocomc`, `run_emcee` or `run_ptlmc` in the same way.